# Results Visualization
Confronto metodi (zero-shot, human_prompt, baseline, APE, APO) su esperimenti **mixed models** e **same models**.

In [ ]:
import json
import os
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import numpy as np
from pathlib import Path

sns.set_theme(style='whitegrid', palette='muted')
METHODS = ['zero_shot', 'human_prompt', 'baseline', 'APE', 'APO']
METHOD_LABELS = {
    'zero_shot':    'Zero-shot',
    'human_prompt': 'Human prompt',
    'baseline':     'Baseline',
    'APE':          'APE',
    'APO':          'APO',
}
ROOT = Path('.')

In [ ]:
def load_json(path):
    with open(path, encoding='utf-8') as f:
        return json.load(f)

def get_test_score(problem, method):
    """Restituisce il test_score per un metodo dato un problema."""
    if method not in problem:
        return None
    m = problem[method]
    if method in ('APE', 'APO'):
        return m.get('best_test')
    return m.get('test_score')

def problems_to_rows(problems, **meta):
    rows = []
    for p in problems:
        base = {
            'problem_idx': p.get('problem_idx'),
            'difficulty':  p.get('difficulty', 'N/A'),
            'task_id':     p.get('task_id', ''),
        }
        base.update(meta)
        for method in METHODS:
            score = get_test_score(p, method)
            if score is not None:
                row = dict(base)
                row['method'] = method
                row['test_score'] = score
                rows.append(row)
    return rows

In [ ]:
# ---------------------------------------------------------------------------
# Configurazione esperimenti
# Per coppie base+resume si usa il file resume (più completo)
# ---------------------------------------------------------------------------
EXPERIMENTS = [
    # Mixed models
    dict(
        file='mixed_models/results/results_20260522_103953.json',
        setup='mixed', dataset='APPS',
        target='Qwen3.5-27B', optimizer='QwenCoder-30B',
        label='mixed\nAPPS\nQ35→QC',
    ),
    dict(
        file='mixed_models/results/results_20260515_182504_resume_20260516_160413.json',
        setup='mixed', dataset='HumanEval',
        target='Qwen3.5-27B', optimizer='QwenCoder-30B',
        label='mixed\nHE\nQ35→QC',
    ),
    dict(
        file='mixed_models/results/results_20260517_134115.json',
        setup='mixed', dataset='APPS',
        target='QwenCoder-30B', optimizer='Qwen3.5-27B',
        label='mixed\nAPPS\nQC→Q35',
    ),
    dict(
        file='mixed_models/results/results_20260520_063948.json',
        setup='mixed', dataset='HumanEval',
        target='QwenCoder-30B', optimizer='Qwen3.5-27B',
        label='mixed\nHE\nQC→Q35',
    ),
    # Same models
    dict(
        file='same_models/results/results_20260522_084655.json',
        setup='same', dataset='HumanEval',
        target='QwenCoder-30B', optimizer='QwenCoder-30B',
        label='same\nHE\nQC',
    ),
    dict(
        file='same_models/results/results_20260522_130841.json',
        setup='same', dataset='APPS',
        target='QwenCoder-30B', optimizer='QwenCoder-30B',
        label='same\nAPPS\nQC',
    ),
    dict(
        file='same_models/results/results_qwen_self_apps_resume_20260529_070803.json',
        setup='same', dataset='APPS',
        target='Qwen3.5-27B', optimizer='Qwen3.5-27B',
        label='same\nAPPS\nQ35',
    ),
]

rows = []
for exp in EXPERIMENTS:
    path = ROOT / exp['file']
    if not path.exists():
        print(f'[SKIP] {path}')
        continue
    problems = load_json(path)
    meta = {k: v for k, v in exp.items() if k != 'file'}
    rows.extend(problems_to_rows(problems, **meta))
    print(f'Loaded {len(problems):3d} problems — {exp["label"].replace(chr(10)," ")}')

df = pd.DataFrame(rows)
df['method_label'] = df['method'].map(METHOD_LABELS)
print(f'\nTotale righe DataFrame: {len(df)}')
df.head()

## 1. Panoramica: score medio per metodo e configurazione

In [ ]:
agg = (
    df.groupby(['label', 'method_label'])['test_score']
    .agg(['mean', 'sem', 'count'])
    .reset_index()
)

method_order = [METHOD_LABELS[m] for m in METHODS if m in df['method'].unique()]
label_order  = [e['label'] for e in EXPERIMENTS if (ROOT / e['file']).exists()]

fig, ax = plt.subplots(figsize=(14, 5))
x = np.arange(len(label_order))
n_methods = len(method_order)
width = 0.8 / n_methods
palette = sns.color_palette('muted', n_methods)

for i, method in enumerate(method_order):
    sub = agg[agg['method_label'] == method].set_index('label')
    means = [sub.loc[lbl, 'mean'] if lbl in sub.index else np.nan for lbl in label_order]
    sems  = [sub.loc[lbl, 'sem']  if lbl in sub.index else 0       for lbl in label_order]
    offset = (i - n_methods / 2 + 0.5) * width
    bars = ax.bar(x + offset, means, width, label=method, color=palette[i],
                  yerr=sems, capsize=3, error_kw=dict(elinewidth=0.8))

ax.set_xticks(x)
ax.set_xticklabels(label_order, fontsize=8)
ax.set_ylim(0, 1.08)
ax.yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
ax.set_ylabel('Test score medio')
ax.set_title('Score medio per metodo e configurazione sperimentale')
ax.legend(title='Metodo', bbox_to_anchor=(1.01, 1), loc='upper left')
plt.tight_layout()
plt.show()

## 2. Mixed vs Same: confronto per dataset e metodo

In [ ]:
agg2 = (
    df.groupby(['setup', 'dataset', 'method_label'])['test_score']
    .agg(['mean', 'sem'])
    .reset_index()
)

datasets = df['dataset'].unique()
fig, axes = plt.subplots(1, len(datasets), figsize=(7 * len(datasets), 5), sharey=True)
if len(datasets) == 1:
    axes = [axes]

for ax, ds in zip(axes, sorted(datasets)):
    sub = agg2[agg2['dataset'] == ds]
    pivot_mean = sub.pivot(index='method_label', columns='setup', values='mean').reindex(method_order)
    pivot_sem  = sub.pivot(index='method_label', columns='setup', values='sem').reindex(method_order)

    x = np.arange(len(method_order))
    setups = ['mixed', 'same']
    colors = ['#4C72B0', '#DD8452']
    w = 0.35
    for j, (setup, color) in enumerate(zip(setups, colors)):
        if setup not in pivot_mean.columns:
            continue
        means = pivot_mean[setup].values
        sems  = pivot_sem[setup].fillna(0).values
        ax.bar(x + (j - 0.5) * w, means, w, label=setup, color=color,
               yerr=sems, capsize=3, error_kw=dict(elinewidth=0.8))

    ax.set_xticks(x)
    ax.set_xticklabels(method_order, rotation=30, ha='right')
    ax.set_title(f'Dataset: {ds}')
    ax.set_ylim(0, 1.08)
    ax.yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
    ax.legend(title='Setup')

axes[0].set_ylabel('Test score medio')
fig.suptitle('Mixed vs Same Models — score medio per metodo', fontsize=13)
plt.tight_layout()
plt.show()

## 3. APPS — breakdown per difficoltà

In [ ]:
apps_df = df[(df['dataset'] == 'APPS') & (df['difficulty'] != 'N/A')].copy()
diff_order = ['introductory', 'interview', 'competition']
diff_present = [d for d in diff_order if d in apps_df['difficulty'].unique()]

agg3 = (
    apps_df.groupby(['difficulty', 'method_label', 'setup'])['test_score']
    .mean()
    .reset_index()
)

fig, axes = plt.subplots(1, len(diff_present), figsize=(6 * len(diff_present), 5), sharey=True)
if len(diff_present) == 1:
    axes = [axes]

for ax, diff in zip(axes, diff_present):
    sub = agg3[agg3['difficulty'] == diff]
    pivot = sub.pivot(index='method_label', columns='setup', values='test_score').reindex(method_order)

    x = np.arange(len(method_order))
    setups = ['mixed', 'same']
    colors = ['#4C72B0', '#DD8452']
    w = 0.35
    for j, (setup, color) in enumerate(zip(setups, colors)):
        if setup not in pivot.columns:
            continue
        ax.bar(x + (j - 0.5) * w, pivot[setup].values, w, label=setup, color=color)

    ax.set_xticks(x)
    ax.set_xticklabels(method_order, rotation=30, ha='right')
    ax.set_title(f'Difficoltà: {diff}')
    ax.set_ylim(0, 1.08)
    ax.yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
    ax.legend(title='Setup')

axes[0].set_ylabel('Test score medio')
fig.suptitle('APPS — score per difficoltà e metodo', fontsize=13)
plt.tight_layout()
plt.show()

## 4. Heatmap: score per configurazione × metodo

In [ ]:
pivot_heat = (
    df.groupby(['label', 'method_label'])['test_score']
    .mean()
    .unstack('method_label')
    .reindex(columns=method_order)
    .reindex(label_order)
)

fig, ax = plt.subplots(figsize=(9, len(label_order) * 0.75 + 1.5))
sns.heatmap(
    pivot_heat,
    annot=True, fmt='.2f', cmap='YlGn',
    vmin=0, vmax=1,
    linewidths=0.5, linecolor='white',
    ax=ax,
)
ax.set_xlabel('Metodo')
ax.set_ylabel('Configurazione')
ax.set_title('Heatmap — test score medio (configurazione × metodo)')
ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=8)
plt.tight_layout()
plt.show()

## 5. Guadagno di APE e APO rispetto al baseline

In [ ]:
# Calcolo delta = APE/APO - baseline per ciascun problema
pivot_prob = (
    df.pivot_table(index=['label', 'problem_idx', 'difficulty', 'setup', 'dataset'],
                   columns='method', values='test_score')
    .reset_index()
)

for opt_method in ['APE', 'APO']:
    if opt_method in pivot_prob.columns and 'baseline' in pivot_prob.columns:
        pivot_prob[f'delta_{opt_method}'] = pivot_prob[opt_method] - pivot_prob['baseline']

delta_cols = [c for c in pivot_prob.columns if c.startswith('delta_')]
if delta_cols:
    delta_df = pivot_prob[['label', 'setup', 'dataset'] + delta_cols].dropna(subset=delta_cols, how='all')
    delta_melt = delta_df.melt(
        id_vars=['label', 'setup', 'dataset'],
        value_vars=delta_cols,
        var_name='method', value_name='delta'
    )
    delta_melt['method'] = delta_melt['method'].str.replace('delta_', '')

    fig, ax = plt.subplots(figsize=(12, 5))
    sns.boxplot(
        data=delta_melt.dropna(subset=['delta']),
        x='label', y='delta', hue='method',
        ax=ax, palette=['#55A868', '#C44E52'],
        order=label_order,
    )
    ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
    ax.set_xticklabels(ax.get_xticklabels(), fontsize=8)
    ax.yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
    ax.set_ylabel('Delta test score (vs baseline)')
    ax.set_title('Guadagno di APE / APO rispetto al baseline (per problema)')
    ax.legend(title='Metodo')
    plt.tight_layout()
    plt.show()
else:
    print('Nessuna colonna delta calcolabile.')

## 6. Tabella riassuntiva

In [ ]:
summary = (
    df.groupby(['setup', 'dataset', 'target', 'method_label'])['test_score']
    .agg(mean='mean', std='std', n='count')
    .reset_index()
)
summary['mean%'] = (summary['mean'] * 100).round(1)
summary['std%']  = (summary['std']  * 100).round(1)
summary = summary.sort_values(['setup', 'dataset', 'target', 'method_label'])

display(
    summary[['setup','dataset','target','method_label','mean%','std%','n']]
    .rename(columns={'method_label':'method','mean%':'mean (%)','std%':'std (%)'})
    .set_index(['setup','dataset','target','method'])
    .style.background_gradient(subset=['mean (%)'], cmap='YlGn', vmin=0, vmax=100)
)